# 03 — Stabilizers and Syndromes

## Learning objectives

This notebook connects the classical parity checks from `02_classical_3bit_repetition_code.ipynb` to their quantum counterparts. By the end you should be able to:

1. explain Pauli $X$ and $Z$, eigenstates and eigenvalues, in plain language;
2. define a *stabilizer* and explain why a code state always has stabilizer eigenvalue $+1$;
3. explain what a *syndrome* is, and why an anticommuting error flips a stabilizer's measured sign;
4. explain why measuring a stabilizer reveals error information without revealing the encoded logical bit;
5. map the classical checks $b_0 \oplus b_1$, $b_1 \oplus b_2$ onto the quantum stabilizers $Z_0Z_1$, $Z_1Z_2$ on the 3-qubit repetition code;
6. reproduce notebook 02's syndrome table using Pauli matrices, and again using a Stim circuit.

**Expected outputs:** a syndrome table (for `no error`, $X_0$, $X_1$, $X_2$) computed two independent ways — Pauli matrix eigenvalues, and a Stim ancilla circuit — that agree with each other and with notebook 02's classical table.

**Assumes:** `01_stim_basics.ipynb` (ancilla-based parity measurement) and `02_classical_3bit_repetition_code.ipynb` (encoding, syndromes, majority-vote decoding) are already understood. Neither notebook is modified here.

**How to use this notebook:** run cells in order. Before a cell marked **Prediction**, stop and write down your own answer first.

## 1. Pauli $X$ and $Z$

A single qubit's state is a length-2 vector. Two operators (2×2 matrices) matter most for this notebook:

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \qquad Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

- $X$ is the **bit flip**: $X|0\rangle = |1\rangle$ and $X|1\rangle = |0\rangle$. It is the quantum version of the classical bit flip used throughout notebook 02.
- $Z$ is the **phase flip**: it leaves $|0\rangle$ and $|1\rangle$ unchanged as *basis states*, but multiplies $|1\rangle$ by $-1$: $Z|0\rangle = +1|0\rangle$, $Z|1\rangle = -1|1\rangle$.

That second property is exactly what we need: $|0\rangle$ and $|1\rangle$ are **eigenstates** of $Z$, with **eigenvalues** $+1$ and $-1$ respectively. An eigenstate of an operator is a state the operator leaves unchanged *up to an overall number* (the eigenvalue) — no rotation into some other state, just a sign or scale.

In [27]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Z = np.array([[1, 0], [0, -1]])
I2 = np.eye(2, dtype=int)

ket0 = np.array([1, 0])
ket1 = np.array([0, 1])

print("Z |0>:", Z @ ket0, " (eigenvalue +1)")
print("Z |1>:", Z @ ket1, " (eigenvalue -1)")
print("X |0>:", X @ ket0, " (bit flip -> |1>)")
print("X |1>:", X @ ket1, " (bit flip -> |0>)")

Z |0>: [1 0]  (eigenvalue +1)
Z |1>: [ 0 -1]  (eigenvalue -1)
X |0>: [0 1]  (bit flip -> |1>)
X |1>: [1 0]  (bit flip -> |0>)


## 2. Stabilizers, syndromes, and why they don't leak the logical bit

**Stabilizer.** For a set of qubits in some state $|\psi\rangle$, an operator $S$ is a *stabilizer* of $|\psi\rangle$ if

$$S|\psi\rangle = +1\,|\psi\rangle.$$

A *code* is defined by picking a small set of stabilizers; the *code space* is whatever set of states all of them stabilize simultaneously (all eigenvalue $+1$). Every valid, error-free codeword lives in that space **by construction** — that is the definition of the code, not something to derive. If you build your code states to be $+1$ eigenstates of $S$, then of course they have eigenvalue $+1$: it's how you chose them.

**Syndrome.** A syndrome is simply the list of measured eigenvalues (one per stabilizer, usually reported as bits: $+1 \to 0$, $-1 \to 1$) after possibly-noisy evolution. It's the direct quantum analogue of running notebook 02's `syndrome()` function on a possibly-corrupted codeword.

**Why an anticommuting error flips the sign.** Suppose an error $E$ *anticommutes* with a stabilizer $S$, meaning $SE = -ES$. If $|\psi\rangle$ is a $+1$ eigenstate of $S$, then after the error occurs the state is $E|\psi\rangle$, and:

$$S\,(E|\psi\rangle) = (SE)|\psi\rangle = (-ES)|\psi\rangle = -E\,(S|\psi\rangle) = -E|\psi\rangle.$$

So $E|\psi\rangle$ is now a $-1$ eigenstate of $S$: measuring $S$ returns $-1$ instead of $+1$. That sign flip *is* the syndrome bit. If instead $E$ *commutes* with $S$ ($SE = ES$), the same calculation gives $+1$ back — $S$ doesn't notice that error at all.

**Why syndromes don't reveal the logical bit.** The *logical* operators of a code (the ones that flip $|0_L\rangle \leftrightarrow |1_L\rangle$, or distinguish them) are deliberately chosen to **commute** with every stabilizer. A stabilizer measurement can only report *whether an error occurred relative to the code space* — it cannot report *which* codeword you started with, because by design it produces the same eigenvalue ($+1$) regardless of which logical value is encoded. Below is a two-qubit-scale demonstration of commuting vs. anticommuting operators, before we scale up to three qubits.

In [28]:
# Two single-qubit operators on DIFFERENT qubits always commute (they act on
# disjoint parts of the tensor-product space). Two Paulis (X, Z) on the SAME
# qubit anticommute. We'll build multi-qubit operators as matrix products of
# single-qubit operators embedded with identities -- this generalizes cleanly
# to 3 qubits below.


def single_qubit_op(op, qubit, n_qubits):
    """Embed a 2x2 single-qubit operator into an n_qubits-qubit space."""
    mats = [I2] * n_qubits
    mats[qubit] = op
    result = mats[0]
    for m in mats[1:]:
        result = np.kron(result, m)
    return result


Z0 = single_qubit_op(Z, 0, n_qubits=2)
X0 = single_qubit_op(X, 0, n_qubits=2)
X1 = single_qubit_op(X, 1, n_qubits=2)

same_qubit_anticommute = np.allclose(Z0 @ X0, -(X0 @ Z0))
different_qubit_commute = np.allclose(Z0 @ X1, X1 @ Z0)

print("Z0 and X0 (same qubit) anticommute:", same_qubit_anticommute)
print("Z0 and X1 (different qubits) commute:", different_qubit_commute)

Z0 and X0 (same qubit) anticommute: True
Z0 and X1 (different qubits) commute: True


## 3. Classical checks ↔ quantum stabilizers

Notebook 02 used two parity checks on a 3-bit codeword $(b_0, b_1, b_2)$:

$$s_0 = b_0 \oplus b_1 \qquad s_1 = b_1 \oplus b_2$$

Their direct quantum counterparts are two-qubit stabilizers built from $Z$:

| Classical check | Quantum stabilizer |
|---|---|
| $b_0 \oplus b_1$ | $Z_0 Z_1$ |
| $b_1 \oplus b_2$ | $Z_1 Z_2$ |

And the bit ↔ eigenvalue correspondence used throughout this notebook is:

| Syndrome bit | Stabilizer eigenvalue | Meaning |
|---:|---:|---|
| `0` | $+1$ | that check is satisfied — no error detected there |
| `1` | $-1$ | that check is violated — an anticommuting error was detected |

This is exactly the classical XOR-parity idea, just phrased as "does the eigenvalue stay $+1$?" instead of "do these two bits agree?". $Z_0Z_1$ applied to a computational basis state $|b_0 b_1 b_2\rangle$ gives eigenvalue $(-1)^{b_0 \oplus b_1}$ — literally the classical parity, exponentiated.

## 4. The 3-qubit repetition code

$$|0_L\rangle = |000\rangle \qquad |1_L\rangle = |111\rangle$$

The stabilizer group is generated by $Z_0Z_1$ and $Z_1Z_2$ (their product $Z_0Z_2$ is the third, non-independent element of the group, plus the identity). Both $|000\rangle$ and $|111\rangle$ are $+1$ eigenstates of both generators — you can check this directly: every pair of neighboring bits in `000` and in `111` agrees, so every parity/eigenvalue is $+1$, matching Section 2's claim that code states are built to have stabilizer eigenvalue $+1$.

In [36]:
def kron_all(*mats):
    result = mats[0]
    for m in mats[1:]:
        result = np.kron(result, m)
    return result


def three_qubit_op(op, qubit):
    return single_qubit_op(op, qubit, n_qubits=3)


Z0Z1 = three_qubit_op(Z, 0) @ three_qubit_op(Z, 1)
Z1Z2 = three_qubit_op(Z, 1) @ three_qubit_op(Z, 2)

state_000 = kron_all(ket0, ket0, ket0)
state_111 = kron_all(ket1, ket1, ket1)

print("Z0Z1 stabilizes |000> and |111>:", Z0Z1.shape, "operator on an 8-dim space")

Z0Z1 stabilizes |000> and |111>: (8, 8) operator on an 8-dim space


### Prediction 1

**Before running the next cell:** what eigenvalue do you expect from $Z_0Z_1$ acting on $|000\rangle$? On $|111\rangle$? Now suppose an $X_0$ error occurs on $|111\rangle$ (turning it into $|011\rangle$) — what eigenvalue do you expect from $Z_0Z_1$ then, and why (tie it back to Section 2's anticommutation argument)?

In [30]:
def eigenvalue(op, state):
    """Return the (real) eigenvalue of `op` on `state`, assuming state is an eigenstate."""
    return float(np.real(state @ (op @ state)))


print("Z0Z1 on |000>:", eigenvalue(Z0Z1, state_000))
print("Z0Z1 on |111>:", eigenvalue(Z0Z1, state_111))

state_011 = three_qubit_op(X, 0) @ state_111  # X0 error applied to |111>
print("Z0Z1 on |111> after X0 error:", eigenvalue(Z0Z1, state_011))

Z0Z1 on |000>: 1.0
Z0Z1 on |111>: 1.0
Z0Z1 on |111> after X0 error: -1.0


As expected, an $X_0$ error flips the $Z_0Z_1$ eigenvalue from $+1$ to $-1$, because $X_0$ anticommutes with $Z_0$ (and therefore with $Z_0Z_1$, since $Z_1$ acts on a different qubit and commutes with everything on qubit 0).

### Prediction 2

The four cases we care about are: no error, $X_0$, $X_1$, $X_2$ (each applied to $|111\rangle$).

**Before running the next cell:** based on which qubits each check touches ($Z_0Z_1$ touches qubits 0 and 1; $Z_1Z_2$ touches qubits 1 and 2), which of the four cases do you expect to share a syndrome with any other case, and which should all be distinct? (Hint: recall from notebook 02 that a distance-3 code must be able to tell all single-bit errors apart.)

In [31]:
def eigen_to_bit(eigenvalue):
    return 0 if eigenvalue > 0 else 1


def matrix_syndrome(state):
    return (eigen_to_bit(eigenvalue(Z0Z1, state)), eigen_to_bit(eigenvalue(Z1Z2, state)))


cases = {
    "no error": state_111,
    "X0": three_qubit_op(X, 0) @ state_111,
    "X1": three_qubit_op(X, 1) @ state_111,
    "X2": three_qubit_op(X, 2) @ state_111,
}

header = f"{'case':<10} {'<Z0Z1>':>8} {'<Z1Z2>':>8} {'syndrome':>10}"
print(header)
print("-" * len(header))
for label, state in cases.items():
    e01 = eigenvalue(Z0Z1, state)
    e12 = eigenvalue(Z1Z2, state)
    s = matrix_syndrome(state)
    print(f"{label:<10} {e01:>8.0f} {e12:>8.0f} {str(s):>10}")

case         <Z0Z1>   <Z1Z2>   syndrome
---------------------------------------
no error          1        1     (0, 0)
X0               -1        1     (1, 0)
X1               -1       -1     (1, 1)
X2                1       -1     (0, 1)


All four syndromes are distinct, and these are exactly the `(s0, s1)` values from notebook 02's table for the same four cases on `encode(1) = 111`: `(0,0)` for no error, `(1,0)` for a flip on qubit 0, `(1,1)` for qubit 1, `(0,1)` for qubit 2. The quantum stabilizer picture and the classical parity picture agree bit-for-bit — as they must, since $Z_0Z_1$ and $Z_1Z_2$ are just $b_0\oplus b_1$ and $b_1\oplus b_2$ written as operators.

## 5. Syndrome bit vs. stabilizer eigenvalue — keep these distinct

It's easy to conflate "the syndrome bit" with "the eigenvalue," but they are related, not identical:

- **Stabilizer eigenvalue** is a physical quantity the stabilizer operator actually has on the state: $+1$ or $-1$.
- **Syndrome bit** is a *label* we choose for bookkeeping and classical decoding: by convention, eigenvalue $+1 \to$ bit `0`, eigenvalue $-1 \to$ bit `1`.

The mapping is a human choice (any consistent labeling would do), whereas the eigenvalue is what the physics/math actually produces. Section 6 measures the eigenvalue via a real (simulated) circuit, and Section 4/`eigen_to_bit` shows the bookkeeping step that turns it into the syndrome bit used for decoding, matching the convention notebook 02 already used for its classical syndromes.

## 6. Measuring stabilizers with Stim

We now build a circuit that measures $Z_0Z_1$ and $Z_1Z_2$ directly, using the exact ancilla technique from notebook 01 (there it was called a "parity measurement" — it's the same circuit, we're now calling it what it is: a stabilizer measurement).

Building it up instruction by instruction:

1. `stim.Circuit()` creates an empty circuit; every qubit starts in $|0\rangle$.
2. To prepare $|1_L\rangle = |111\rangle$, apply `X` to qubits 0, 1, 2.
3. To inject a bit-flip error, apply `X` to whichever data qubit(s) should flip.
4. To measure $Z_0Z_1$: use a fresh ancilla qubit (qubit 3), `CX` from qubit 0 into the ancilla, then `CX` from qubit 1 into the ancilla, then measure the ancilla with `M`. This is exactly notebook 01's two-CNOT parity trick, applied to data qubits 0 and 1.
5. To measure $Z_1Z_2$: a second ancilla (qubit 4), `CX` from qubit 1, `CX` from qubit 2, then `M`.

Putting it together into one circuit-building function:

In [32]:
import stim


def syndrome_circuit(logical_bit, error_qubits=()):
    """Prepare |logical_bit>_L on qubits 0-2, inject X errors, measure Z0Z1 and Z1Z2.

    Ancilla 3 reports the Z0Z1 syndrome bit; ancilla 4 reports the Z1Z2 syndrome bit.
    """
    circuit = stim.Circuit()

    if logical_bit == 1:
        circuit.append("X", [0, 1, 2])

    for qubit in error_qubits:
        circuit.append("X", [qubit])

    # Z0 Z1 via ancilla 3 (same two-CNOT pattern as notebook 01).
    circuit.append("CX", [0, 3])
    circuit.append("CX", [1, 3])

    # Z1 Z2 via ancilla 4.
    circuit.append("CX", [1, 4])
    circuit.append("CX", [2, 4])

    circuit.append("M", [3, 4])
    return circuit


print(syndrome_circuit(logical_bit=1, error_qubits=(0,)))

X 0 1 2 0
CX 0 3 1 3 1 4 2 4
M 3 4


### Prediction 3

**Before running the next cell:** for logical bit 1 with an $X_1$ error, what do you expect the two ancilla measurements (Z0Z1 syndrome, Z1Z2 syndrome) to be? Compare with the `X1` row from Section 4's table.

In [33]:
stim_cases = {
    "no error": (),
    "X0": (0,),
    "X1": (1,),
    "X2": (2,),
}

header = f"{'case':<10} {'stim syndrome':>15} {'matrix syndrome':>17} {'match':>7}"
print(header)
print("-" * len(header))
for label, error_qubits in stim_cases.items():
    circuit = syndrome_circuit(logical_bit=1, error_qubits=error_qubits)
    stim_bits = circuit.compile_sampler().sample(shots=1)[0]
    stim_syndrome = (int(stim_bits[0]), int(stim_bits[1]))
    expected = matrix_syndrome(cases[label])
    print(f"{label:<10} {str(stim_syndrome):>15} {str(expected):>17} {str(stim_syndrome == expected):>7}")

case         stim syndrome   matrix syndrome   match
----------------------------------------------------
no error            (0, 0)            (0, 0)    True
X0                  (1, 0)            (1, 0)    True
X1                  (1, 1)            (1, 1)    True
X2                  (0, 1)            (0, 1)    True


The Stim ancilla measurements reproduce the matrix-based syndromes exactly, which in turn reproduce notebook 02's classical syndrome table. Three independent computations — classical XOR, Pauli-matrix eigenvalues, and a simulated quantum circuit — all agree, because they're all describing the same underlying idea.

## 7. TODO exercises

These are for you to complete — the cells below intentionally only contain a stub. Fill them in, then run them.

**TODO 1:** $Z_0Z_2$ is the third (non-independent) element of the stabilizer group: $Z_0Z_2 = (Z_0Z_1)(Z_1Z_2)$, since $Z_1 Z_1 = I$. Write `third_stabilizer_eigenvalue(state)` that builds $Z_0Z_2$ from `three_qubit_op` and returns its eigenvalue on `state`. Then confirm, for each of the four cases in `cases`, that this eigenvalue equals the *product* of the $Z_0Z_1$ and $Z_1Z_2$ eigenvalues you already computed in Section 4.

In [41]:
def third_stabilizer_eigenvalue(state):
    """Return the eigenvalue of Z0Z2 on `state`, assuming state is an eigenstate."""
    Z0Z2 = three_qubit_op(Z, 0) @ three_qubit_op(Z, 2)
    return eigenvalue(Z0Z2, state)


# TODO: once implemented, check it against the product of the other two eigenvalues, e.g.:
for label, state in cases.items():
    predicted = eigenvalue(Z0Z1, state) * eigenvalue(Z1Z2, state)
    actual = third_stabilizer_eigenvalue(state)
    print(label, predicted, actual, predicted == actual)

no error 1.0 1.0 True
X0 -1.0 -1.0 True
X1 1.0 1.0 True
X2 -1.0 -1.0 True


**TODO 2:** Section 2 showed, at 2-qubit scale, that same-qubit Paulis anticommute while different-qubit Paulis commute. Now verify the claim from Section 2 that matters for *this* code: the logical operator $X_L = X_0X_1X_2$ (which flips $|0_L\rangle \leftrightarrow |1_L\rangle$) **commutes** with the stabilizer $Z_0Z_1$ — i.e. measuring the stabilizer cannot disturb or reveal the logical value. Write `commutator_is_zero(op_a, op_b)` that returns `True` if two matrices commute (hint: compare `op_a @ op_b` and `op_b @ op_a` with `np.allclose`), then apply it to $X_L$ (built from `three_qubit_op(X, i)` for `i` in `0, 1, 2`) and $Z_0Z_1$.

In [42]:
def commutator_is_zero(op_a, op_b):
    """TODO: return True if op_a and op_b commute (op_a @ op_b == op_b @ op_a)."""
    commutator = op_a @ op_b - op_b @ op_a
    if np.allclose(commutator, np.zeros_like(commutator)):
        return True
    else:
        return False


# TODO: once implemented, try e.g.:
X_L = three_qubit_op(X, 0) @ three_qubit_op(X, 1) @ three_qubit_op(X, 2)
print(commutator_is_zero(X_L, Z0Z1))

True


## 8. Closed-notebook recall checklist

Close this notebook (or at least stop scrolling up) and answer these from memory:

1. What does it mean for a state to be a "$+1$ eigenstate" of an operator?
2. In your own words, what is a stabilizer, and why does a code state have stabilizer eigenvalue $+1$?
3. Why does an error that *anticommutes* with a stabilizer flip its measured eigenvalue, while a commuting error does not?
4. Why can a stabilizer measurement reveal that an error occurred without revealing which logical value ($0_L$ or $1_L$) was encoded?
5. Which classical parity check corresponds to $Z_1Z_2$, and what is the syndrome-bit convention ($0$/$1$) for eigenvalues $+1$/$-1$?

If any of these feel shaky, re-read the matching section above before moving on.